# 03. 모델 평가와 심층 분석

이번 노트북의 목적은 학습을 더 하는 것이 아니라, **저장된 best model을 같은 test set에서 공정하게 비교하고 분석하는 것**입니다.

특히 과제 요구사항인 accuracy, confusion matrix, precision, recall을 계산하고, 어떤 상황에서 어떤 지표가 더 중요한지 제 생각을 정리합니다.

최종 분석 대상은 02-2에서 만든 두 모델입니다.

- **Regularized Original:** 원본 train 데이터 + 규제 모델
- **Regularized OpenCV:** OpenCV 증강 train 데이터 + 같은 규제 모델

저는 성능과 효율을 함께 고려해서 **Regularized Original을 최종 선택 후보**로 보고, OpenCV 모델은 비교 후보로 사용합니다.

## Step 0. 분석 목표를 정리합니다

02 단계까지는 여러 모델을 학습하면서 train loss와 validation loss를 비교했습니다.

03 단계에서는 다음 질문에 답합니다.

1. validation 결과를 기준으로 왜 두 모델을 최종 분석 후보로 골랐는가?
2. test set에서 두 모델의 accuracy는 어떻게 다른가?
3. confusion matrix를 보면 어떤 방향의 오분류가 많은가?
4. precision과 recall은 각각 어떤 상황에서 더 중요한가?
5. 실제 오분류 이미지는 어떤 특징을 가지고 있는가?

## Step 1. 필요한 패키지와 경로를 준비합니다

이번 노트북에서는 모델을 새로 학습하지 않습니다. 02-2에서 저장한 checkpoint를 불러와서 test set에만 사용합니다.

Keras/TensorFlow로 비유하면 `model.fit()`을 다시 하는 것이 아니라, 저장된 모델을 `load_model()`로 불러와 `model.evaluate()`와 `model.predict()`를 하는 단계입니다.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from PIL import Image
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    classification_report,
    confusion_matrix,
    precision_recall_fscore_support,
)
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms

PROJECT_DIR = Path.cwd()
if PROJECT_DIR.name == "notebooks":
    PROJECT_DIR = PROJECT_DIR.parent

PROCESSED_DIR = PROJECT_DIR / "data" / "processed"
METRICS_DIR = PROJECT_DIR / "outputs" / "metrics"
FIGURE_DIR = PROJECT_DIR / "outputs" / "figures"
CHECKPOINT_DIR = PROJECT_DIR / "outputs" / "checkpoints"
MISCLASSIFIED_DIR = PROJECT_DIR / "outputs" / "misclassified"

for directory in [METRICS_DIR, FIGURE_DIR, CHECKPOINT_DIR, MISCLASSIFIED_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

SPLIT_CSV_PATH = PROCESSED_DIR / "pet_binary_split_70_15_15_breed_stratified.csv"

ORIGINAL_CHECKPOINT_PATH = CHECKPOINT_DIR / "best_03_regularized_original_cnn.pt"
OPENCV_CHECKPOINT_PATH = CHECKPOINT_DIR / "best_06_final_opencv_regularized_cnn.pt"

IMAGE_SIZE = 128
BATCH_SIZE = 32
LABEL_TO_INDEX = {"cat": 0, "dog": 1}
INDEX_TO_LABEL = {0: "cat", 1: "dog"}

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

## Step 2. 저장된 학습 summary를 한 표로 모읍니다

먼저 모든 실험 결과를 한눈에 보기 위해 summary 파일을 불러옵니다.

여기서 중요한 점은 **가장 높은 숫자 하나만 보고 결정하지 않는 것**입니다. validation loss, validation accuracy, 학습 시간, 데이터셋 특성을 함께 봅니다.

In [ ]:
def add_summary_row(rows, model_name, data_name, improvement, best_val_loss, best_val_accuracy, final_val_accuracy, elapsed_time_seconds, checkpoint_path, decision):
    """여러 summary 파일의 컬럼 이름이 조금 달라도 같은 비교표에 넣기 위한 helper 함수입니다."""
    rows.append({
        "model_name": model_name,
        "data": data_name,
        "improvement": improvement,
        "best_val_loss": best_val_loss,
        "best_val_accuracy": best_val_accuracy,
        "final_val_accuracy": final_val_accuracy,
        "elapsed_time_seconds": elapsed_time_seconds,
        "checkpoint_path": checkpoint_path,
        "decision": decision,
    })


summary_rows = []

# 02, 02-1, 02-2에서 저장한 모든 학습 summary를 한 파일에서 읽습니다.
# 이 파일에는 기본 3 epoch 모델, EarlyStopping baseline, OpenCV-only, 규제 원본, 규제 OpenCV가 모두 들어갑니다.
all_summary_path = METRICS_DIR / "all_training_summaries.csv"
all_summary_df = pd.read_csv(all_summary_path)

experiment_labels = {
    "step13_basic_3_epoch": {
        "model_name": "Baseline 3 Epoch",
        "data": "Original train",
        "improvement": "기본 CNN",
        "decision": "기준 모델",
    },
    "step13_1_improved_early_stopping": {
        "model_name": "Baseline EarlyStopping",
        "data": "Original train",
        "improvement": "EarlyStopping + checkpoint",
        "decision": "02 개선 baseline",
    },
    "opencv_augmented_simplecnn": {
        "model_name": "OpenCV SimpleCNN",
        "data": "OpenCV augmented train",
        "improvement": "증강만 추가",
        "decision": "증강만으로는 부족",
    },
    "regularized_original_train": {
        "model_name": "Regularized Original",
        "data": "Original train",
        "improvement": "규제 모델",
        "decision": "최종 선택 후보",
    },
    "regularized_opencv_augmented_train": {
        "model_name": "Regularized OpenCV",
        "data": "OpenCV augmented train",
        "improvement": "규제 + 증강",
        "decision": "비교 후보",
    },
}

for experiment_name, label_info in experiment_labels.items():
    matched_rows = all_summary_df[all_summary_df["experiment_name"] == experiment_name]
    if matched_rows.empty:
        print(f"summary에 없습니다: {experiment_name}")
        continue

    row = matched_rows.iloc[0]
    add_summary_row(
        summary_rows,
        label_info["model_name"],
        label_info["data"],
        label_info["improvement"],
        float(row["best_val_loss"]),
        float(row["best_val_accuracy"]),
        float(row.get("final_val_accuracy", np.nan)),
        float(row["elapsed_time_seconds"]),
        row.get("checkpoint_path", ""),
        label_info["decision"],
    )

model_summary_df = pd.DataFrame(summary_rows)
model_summary_df

## Step 3. 막대 그래프로 전체 실험을 비교합니다

막대 그래프는 발표에서 좋습니다. 숫자 표만 보여주면 청중이 바로 비교하기 어렵지만, 그래프는 어떤 실험이 개선되었는지 한눈에 보입니다.

단, loss, accuracy, 학습 시간은 단위가 다르기 때문에 한 그래프에 섞지 않고 따로 그립니다.

In [ ]:
plot_df = model_summary_df.dropna(subset=["best_val_loss"]).copy()

colors = ["#4C78A8" if name == "Regularized Original" else "#72B7B2" if name == "Regularized OpenCV" else "#B0B0B0" for name in plot_df["model_name"]]

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].bar(plot_df["model_name"], plot_df["best_val_loss"], color=colors)
axes[0].set_title("Best Validation Loss")
axes[0].set_ylabel("loss, lower is better")
axes[0].tick_params(axis="x", rotation=25)

axes[1].bar(plot_df["model_name"], plot_df["best_val_accuracy"], color=colors)
axes[1].set_title("Best Validation Accuracy")
axes[1].set_ylabel("accuracy, higher is better")
axes[1].set_ylim(0, 1)
axes[1].tick_params(axis="x", rotation=25)

axes[2].bar(plot_df["model_name"], plot_df["elapsed_time_seconds"], color=colors)
axes[2].set_title("Training Time")
axes[2].set_ylabel("seconds, lower is more efficient")
axes[2].tick_params(axis="x", rotation=25)

plt.tight_layout()
summary_bar_path = FIGURE_DIR / "model_selection_summary_bars.png"
plt.savefig(summary_bar_path, dpi=150, bbox_inches="tight")
plt.show()

summary_bar_path

## Step 4. 두 모델을 선택한 이유를 정리합니다

이번 최종 분석에서는 **Regularized Original**을 최종 선택 후보로 보고, **Regularized OpenCV**를 비교 후보로 둡니다.

선택 이유는 다음과 같습니다.

- OpenCV 모델은 best validation loss가 조금 더 낮습니다.
- 하지만 validation accuracy 차이는 크지 않고, 학습 시간은 훨씬 깁니다.
- Oxford-IIIT Pet Dataset은 비교적 잘 찍힌 반려동물 사진이 많기 때문에, 조명 변화나 blur 같은 OpenCV 증강이 test 분포와 크게 맞지 않았을 수 있습니다.
- 따라서 이번 과제에서는 성능과 효율의 균형이 좋은 Regularized Original을 최종 선택 후보로 삼고, OpenCV 모델이 실제 test set에서 도움이 되는지는 비교로 확인합니다.

발표에서는 이렇게 말할 수 있습니다.

> OpenCV 증강 모델의 best validation loss가 조금 더 낮았지만, 성능 차이가 크지 않았고 학습 시간은 훨씬 길었습니다. 또한 현재 데이터셋은 비교적 좋은 조건에서 찍힌 이미지가 많기 때문에 OpenCV 증강이 큰 효과를 내지 못했을 가능성이 있다고 판단했습니다. 그래서 성능과 효율을 함께 고려해 Regularized Original 모델을 최종 선택 후보로 두고, OpenCV 모델은 비교 후보로 분석했습니다.

## Step 5. test set을 불러옵니다

test set은 학습과 validation에 사용하지 않은 마지막 평가 데이터입니다.

validation set은 학습 중 모델 선택과 early stopping에 참고했습니다. 반면 test set은 최종 확인용으로만 사용합니다.

In [ ]:
split_df = pd.read_csv(SPLIT_CSV_PATH)
test_df = split_df[split_df["split"] == "test"].reset_index(drop=True)

test_df[["label", "breed"]].value_counts().reset_index(name="count").head(10)

## Step 6. test용 Dataset과 DataLoader를 만듭니다

학습 때와 같은 이미지 전처리를 사용해야 합니다. 학습 때 128x128 resize, ToTensor, Normalize를 사용했기 때문에 test에서도 같은 기준을 적용합니다.

In [ ]:
test_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5]),
])


class PetImageDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        self.dataframe = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, index):
        row = self.dataframe.iloc[index]

        # 이미지는 RGB로 통일합니다. 흑백이나 다른 모드 이미지가 섞여 있어도 모델 입력을 3채널로 맞추기 위해서입니다.
        image = Image.open(row["image_path"]).convert("RGB")
        if self.transform is not None:
            image = self.transform(image)

        label_index = LABEL_TO_INDEX[row["label"]]
        return image, label_index, row["image_path"], row["label"], row["breed"]


test_dataset = PetImageDataset(test_df, transform=test_transform)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

len(test_dataset)

## Step 7. 02-2와 같은 모델 구조를 정의합니다

checkpoint는 가중치만 저장되어 있기 때문에, 모델 구조가 학습 때와 같아야 합니다.

여기서는 02-2에서 사용한 `RegularizedCNN` 구조를 그대로 다시 정의합니다.

In [ ]:
class RegularizedCNN(nn.Module):
    def __init__(self, dropout_p=0.5):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(in_channels=3, out_channels=16, kernel_size=3, padding=1),
            nn.BatchNorm2d(16),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2),

            nn.Conv2d(in_channels=16, out_channels=32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2),

            nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2),
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(p=dropout_p),
            nn.Linear(64 * 16 * 16, 64),
            nn.ReLU(),
            nn.Dropout(p=dropout_p),
            nn.Linear(64, 2),
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

## Step 8. 두 checkpoint를 불러와 test set 예측을 만듭니다

여기서는 `model.eval()`과 `torch.no_grad()`를 사용합니다.

- `model.eval()`: Dropout과 BatchNorm을 평가 모드로 바꿉니다.
- `torch.no_grad()`: 평가 단계이므로 gradient를 계산하지 않습니다.

즉, test 단계에서는 모델 가중치가 업데이트되지 않습니다.

In [ ]:
def load_regularized_model(checkpoint_path):
    model = RegularizedCNN(dropout_p=0.5).to(device)
    checkpoint = torch.load(checkpoint_path, map_location=device)
    model.load_state_dict(checkpoint["model_state_dict"])
    model.eval()
    return model, checkpoint


def predict_on_test(model, loader, model_name):
    prediction_rows = []

    with torch.no_grad():
        for images, labels, image_paths, true_labels, breeds in loader:
            images = images.to(device)
            labels = labels.to(device)

            logits = model(images)
            probabilities = torch.softmax(logits, dim=1)
            predicted_indices = torch.argmax(probabilities, dim=1)

            for i in range(len(labels)):
                true_index = int(labels[i].cpu().item())
                predicted_index = int(predicted_indices[i].cpu().item())
                confidence = float(probabilities[i, predicted_index].cpu().item())

                prediction_rows.append({
                    "model_name": model_name,
                    "image_path": image_paths[i],
                    "breed": breeds[i],
                    "true_label": INDEX_TO_LABEL[true_index],
                    "predicted_label": INDEX_TO_LABEL[predicted_index],
                    "true_index": true_index,
                    "predicted_index": predicted_index,
                    "cat_probability": float(probabilities[i, LABEL_TO_INDEX["cat"]].cpu().item()),
                    "dog_probability": float(probabilities[i, LABEL_TO_INDEX["dog"]].cpu().item()),
                    "predicted_confidence": confidence,
                    "is_correct": true_index == predicted_index,
                })

    return pd.DataFrame(prediction_rows)


model_configs = [
    {
        "model_name": "Regularized Original",
        "checkpoint_path": ORIGINAL_CHECKPOINT_PATH,
        "role": "최종 선택 후보",
    },
    {
        "model_name": "Regularized OpenCV",
        "checkpoint_path": OPENCV_CHECKPOINT_PATH,
        "role": "비교 후보",
    },
]

all_prediction_dfs = []
for config in model_configs:
    model, checkpoint = load_regularized_model(config["checkpoint_path"])
    prediction_df = predict_on_test(model, test_loader, config["model_name"])
    prediction_df["role"] = config["role"]
    all_prediction_dfs.append(prediction_df)

predictions_df = pd.concat(all_prediction_dfs, ignore_index=True)
predictions_path = METRICS_DIR / "test_predictions_two_regularized_models.csv"
predictions_df.to_csv(predictions_path, index=False, encoding="utf-8-sig")

predictions_df.head()

## Step 9. accuracy, precision, recall을 계산합니다

각 지표의 의미는 다음과 같습니다.

- **accuracy:** 전체 test 이미지 중 맞힌 비율입니다.
- **precision:** 모델이 특정 class라고 예측한 것 중 실제로 맞은 비율입니다.
- **recall:** 실제 특정 class인 것 중 모델이 놓치지 않고 찾아낸 비율입니다.
- **f1-score:** precision과 recall의 균형을 보는 지표입니다.

cat/dog 데이터가 비교적 균형적이면 accuracy도 의미가 있습니다. 하지만 어떤 class를 더 자주 틀리는지 보려면 precision과 recall이 필요합니다.

In [ ]:
metric_rows = []

for model_name, group_df in predictions_df.groupby("model_name"):
    y_true = group_df["true_index"].to_numpy()
    y_pred = group_df["predicted_index"].to_numpy()

    accuracy = accuracy_score(y_true, y_pred)
    precision_macro, recall_macro, f1_macro, _ = precision_recall_fscore_support(
        y_true,
        y_pred,
        average="macro",
        zero_division=0,
    )

    class_report = classification_report(
        y_true,
        y_pred,
        labels=[0, 1],
        target_names=["cat", "dog"],
        output_dict=True,
        zero_division=0,
    )

    metric_rows.append({
        "model_name": model_name,
        "accuracy": accuracy,
        "macro_precision": precision_macro,
        "macro_recall": recall_macro,
        "macro_f1": f1_macro,
        "cat_precision": class_report["cat"]["precision"],
        "cat_recall": class_report["cat"]["recall"],
        "dog_precision": class_report["dog"]["precision"],
        "dog_recall": class_report["dog"]["recall"],
    })

test_metrics_df = pd.DataFrame(metric_rows).sort_values("accuracy", ascending=False)
test_metrics_path = METRICS_DIR / "test_metrics_two_regularized_models.csv"
test_metrics_df.to_csv(test_metrics_path, index=False, encoding="utf-8-sig")

test_metrics_df

## Step 10. confusion matrix를 확인합니다

confusion matrix는 모델이 **어떤 방향으로 틀렸는지** 보여줍니다.

예를 들어 cat을 dog로 많이 틀렸다면, 고양이 사진 중 강아지처럼 보이는 패턴이 있는지 오분류 이미지를 봐야 합니다.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, (model_name, group_df) in zip(axes, predictions_df.groupby("model_name")):
    y_true = group_df["true_index"].to_numpy()
    y_pred = group_df["predicted_index"].to_numpy()

    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["cat", "dog"])
    disp.plot(ax=ax, cmap="Blues", colorbar=False, values_format="d")
    ax.set_title(model_name)

plt.tight_layout()
confusion_matrix_path = FIGURE_DIR / "test_confusion_matrices_two_regularized_models.png"
plt.savefig(confusion_matrix_path, dpi=150, bbox_inches="tight")
plt.show()

confusion_matrix_path

## Step 11. 오분류 이미지를 저장하고 직접 확인합니다

숫자 지표만 보면 왜 틀렸는지 알기 어렵습니다. 그래서 실제로 틀린 이미지를 확인합니다.

확인할 때는 다음 관점으로 봅니다.

- 사진이 어둡거나 흐린가?
- 얼굴이 일부만 보이는가?
- 배경이 복잡한가?
- 품종 특성 때문에 cat/dog 경계가 애매한가?
- 모델이 높은 확신으로 틀렸는가?

In [ ]:
for model_name, group_df in predictions_df.groupby("model_name"):
    misclassified_df = group_df[group_df["is_correct"] == False].copy()
    safe_model_name = model_name.lower().replace(" ", "_")
    misclassified_path = MISCLASSIFIED_DIR / f"{safe_model_name}_misclassified.csv"
    misclassified_df.to_csv(misclassified_path, index=False, encoding="utf-8-sig")
    print(model_name, "misclassified count:", len(misclassified_df), "saved to", misclassified_path)


def show_misclassified_samples(prediction_dataframe, model_name, max_images=8):
    sample_df = prediction_dataframe[
        (prediction_dataframe["model_name"] == model_name) &
        (prediction_dataframe["is_correct"] == False)
    ].sort_values("predicted_confidence", ascending=False).head(max_images)

    if sample_df.empty:
        print(f"{model_name}: 오분류 이미지가 없습니다.")
        return

    cols = 4
    rows = int(np.ceil(len(sample_df) / cols))
    fig, axes = plt.subplots(rows, cols, figsize=(16, 4 * rows))
    axes = np.array(axes).reshape(-1)

    for ax, (_, row) in zip(axes, sample_df.iterrows()):
        image = Image.open(row["image_path"]).convert("RGB")
        ax.imshow(image)
        ax.axis("off")
        ax.set_title(
            f"true: {row['true_label']} / pred: {row['predicted_label']}\n"
            f"conf: {row['predicted_confidence']:.2f} / breed: {row['breed']}",
            fontsize=10,
        )

    for ax in axes[len(sample_df):]:
        ax.axis("off")

    plt.suptitle(f"High-confidence misclassified samples - {model_name}", fontsize=14)
    plt.tight_layout()
    plt.show()


show_misclassified_samples(predictions_df, "Regularized Original", max_images=8)
show_misclassified_samples(predictions_df, "Regularized OpenCV", max_images=8)

## Step 12. 지표별로 어떤 상황에서 중요한지 정리합니다

이번 과제에서는 cat/dog 데이터 비율이 비교적 균형적이기 때문에 accuracy가 기본 비교 지표로 의미가 있습니다.

하지만 accuracy만으로는 어떤 class에서 문제가 생겼는지 알 수 없습니다.

- **accuracy가 중요한 상황:** 전체적으로 모델이 얼마나 잘 맞히는지 빠르게 비교할 때 중요합니다.
- **confusion matrix가 중요한 상황:** cat을 dog로 틀리는지, dog를 cat으로 틀리는지처럼 오분류 방향을 보고 싶을 때 중요합니다.
- **precision이 중요한 상황:** 모델이 dog라고 예측한 결과를 믿어도 되는지가 중요할 때 사용합니다. 예를 들어 자동 태깅 시스템에서 잘못된 태그를 줄이고 싶다면 precision이 중요합니다.
- **recall이 중요한 상황:** 실제 dog 이미지를 최대한 놓치지 않는 것이 중요할 때 사용합니다. 예를 들어 특정 동물 이미지를 빠짐없이 수집해야 한다면 recall이 중요합니다.

이번 프로젝트에서는 단순 정답률만 보지 않고, confusion matrix와 precision/recall을 함께 확인해서 모델이 어떤 방식으로 실패하는지 파악하는 것이 더 중요하다고 판단했습니다.

## Step 13. 최종 해석 문장 초안

아래 문장은 결과 수치를 확인한 뒤 발표 자료에 맞게 조금 수정해서 사용합니다.

> 여러 학습 실험을 비교한 결과, OpenCV 증강 모델은 validation loss가 조금 더 낮았지만 학습 시간이 크게 증가했고 accuracy 차이는 크지 않았습니다. Oxford-IIIT Pet Dataset은 비교적 좋은 조건에서 촬영된 이미지가 많아 조명 변화나 blur 중심의 증강이 큰 효과를 내지 못했을 가능성이 있다고 판단했습니다. 따라서 성능과 효율의 균형이 좋은 Regularized Original 모델을 최종 선택 후보로 두고, OpenCV 모델은 비교 후보로 test set에서 함께 평가했습니다. 최종 평가는 accuracy뿐 아니라 confusion matrix, precision, recall, 오분류 이미지를 함께 확인하여 모델이 어떤 상황에서 실패하는지 분석했습니다.